### Rotina para converter formato de dados da estação da CETESB para o formato da opeanAQ

In [1]:
%matplotlib widget

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1. Abre CETESB

In [3]:
cetesb = pd.read_csv("../data/operational/cetesb_20260710.csv",sep=";",skiprows=8,names=["data", "hora", "pm25"],encoding="latin1")
cetesb.head()

,data,hora,pm25
0,10/06/2026,01:00,62.0
1,10/06/2026,02:00,66.0
2,10/06/2026,03:00,59.0
3,10/06/2026,04:00,63.0
4,10/06/2026,05:00,63.0


In [4]:
cetesb["data"] = cetesb["data"].astype(str).str.strip()
cetesb["hora"] = cetesb["hora"].astype(str).str.strip()

# usa 24h em vez de 00h
mask_24 = cetesb["hora"].eq("24:00")
cetesb.loc[mask_24, "hora"] = "00:00"

In [5]:
cetesb["time"] = pd.to_datetime(
    cetesb["data"] + " " + cetesb["hora"],
    format="%d/%m/%Y %H:%M"
)

In [6]:
cetesb.head()

,data,hora,pm25,time
0,10/06/2026,01:00,62.0,2026-06-10 01:00:00
1,10/06/2026,02:00,66.0,2026-06-10 02:00:00
2,10/06/2026,03:00,59.0,2026-06-10 03:00:00
3,10/06/2026,04:00,63.0,2026-06-10 04:00:00
4,10/06/2026,05:00,63.0,2026-06-10 05:00:00


In [7]:
# soma 1 dia nas linhas que eram 24:00
cetesb.loc[mask_24, "time"] = cetesb.loc[mask_24, "time"] + pd.Timedelta(days=1)

In [8]:
# São Paulo -> UTC
cetesb["time"] = (
    cetesb["time"]
    .dt.tz_localize("America/Sao_Paulo")
    .dt.tz_convert("UTC")
)

# mantém só time e pm25 para manter formato do opeanAQ
cetesb = cetesb[["time", "pm25"]]

cetesb.head()

,time,pm25
0,2026-06-10 04:00:00+00:00,62.0
1,2026-06-10 05:00:00+00:00,66.0
2,2026-06-10 06:00:00+00:00,59.0
3,2026-06-10 07:00:00+00:00,63.0
4,2026-06-10 08:00:00+00:00,63.0


In [9]:
# garante ordenação e remove duplicatas, se houver
cetesb["time"] = pd.to_datetime(cetesb["time"], utc=True)
cetesb["pm25"] = pd.to_numeric(cetesb["pm25"], errors="coerce")

cetesb = (
    cetesb
    .sort_values("time")
    .drop_duplicates(subset="time", keep="first")
)

# cria grade horária completa
time_full = pd.date_range(
    start=cetesb["time"].min(),
    end=cetesb["time"].max(),
    freq="h",
    tz="UTC"
)

# reindexa e preenche timestamps faltantes com NaN
cetesb = (
    cetesb
    .set_index("time")
    .reindex(time_full)
    .rename_axis("time")
    .reset_index()
)

# mantém formato final
cetesb = cetesb[["time", "pm25"]]

cetesb.head()

,time,pm25
0,2026-06-10 04:00:00+00:00,62.0
1,2026-06-10 05:00:00+00:00,66.0
2,2026-06-10 06:00:00+00:00,59.0
3,2026-06-10 07:00:00+00:00,63.0
4,2026-06-10 08:00:00+00:00,63.0


In [10]:
cetesb["time"].diff().value_counts()

time
0 days 01:00:00    743
Name: count, dtype: int64

In [ ]:
cetesb.to_csv("../data/operational/openaq_location_6139516_20260709.csv", index=False)